In [3]:
from pathlib import Path
import os

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "config.yaml").is_file() and (candidate / "params.yaml").is_file():
            return candidate

    # Fallback for this project's known local location.
    fallback = Path(r"D:\Somesh\NLP_Project")
    if (fallback / "config" / "config.yaml").is_file() and (fallback / "params.yaml").is_file():
        return fallback

    raise FileNotFoundError(
        "Could not find project root containing config/config.yaml and params.yaml. "
        f"Started searching from: {start}"
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

CONFIG_FILE_PATH = PROJECT_ROOT / "config" / "config.yaml"
PARAMS_FILE_PATH = PROJECT_ROOT / "params.yaml"

print("Project root:", PROJECT_ROOT)
print("Config exists:", CONFIG_FILE_PATH.is_file())
print("Params exists:", PARAMS_FILE_PATH.is_file())


Project root: D:\Somesh\NLP_Project
Config exists: True
Params exists: True


In [4]:
%pwd

'D:\\Somesh\\NLP_Project'

In [5]:
# Keep all relative project paths anchored to the detected project root.
os.chdir(PROJECT_ROOT)


In [6]:
%pwd

'D:\\Somesh\\NLP_Project'

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int


In [8]:
from NLP_Project.logging import logger
from NLP_Project.utils.common import read_yaml, create_directories

# Use absolute paths so notebook working-directory changes cannot break config loading.
CONFIG_FILE_PATH = PROJECT_ROOT / "config" / "config.yaml"
PARAMS_FILE_PATH = PROJECT_ROOT / "params.yaml"


In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.get("TrainingArguments", {})

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_ckpt=str(config.model_ckpt),
            num_train_epochs=int(params.get("num_train_epochs", 1)),
            warmup_steps=int(params.get("warmup_steps", 0)),
            per_device_train_batch_size=int(params.get("per_device_train_batch_size", 1)),
            weight_decay=float(params.get("weight_decay", 0.01)),
            logging_steps=int(params.get("logging_steps", 10)),
            evaluation_strategy=str(params.get("evaluation_strategy", "steps")),
            eval_steps=int(params.get("eval_steps", 500)),
            save_steps=int(params.get("save_steps", 1000)),
            gradient_accumulation_steps=int(params.get("gradient_accumulation_steps", 1)),
        )

        return model_trainer_config


In [10]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from datasets import load_from_disk
import torch


d:\Somesh\NLP_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        # Validate input paths before loading large model files.
        data_path = Path(self.config.data_path)
        if not data_path.exists():
            raise FileNotFoundError(f"Transformed dataset not found: {data_path}")

        output_dir = Path(self.config.root_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        logger.info(f"Loading tokenizer and model: {self.config.model_ckpt}")
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt)

        logger.info(f"Loading transformed dataset: {data_path}")
        dataset = load_from_disk(str(data_path))

        required_splits = {"train", "validation"}
        available_splits = set(dataset.keys()) if hasattr(dataset, "keys") else set()
        missing_splits = required_splits - available_splits
        if missing_splits:
            raise ValueError(
                f"Dataset is missing required split(s): {sorted(missing_splits)}. "
                f"Available splits: {sorted(available_splits)}"
            )

        data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model,
        )

        strategy = self.config.evaluation_strategy.lower()
        if strategy not in {"no", "steps", "epoch"}:
            logger.warning(
                f"Unsupported evaluation_strategy={strategy!r}; defaulting to 'steps'."
            )
            strategy = "steps"

        # Avoid evaluation arguments that are invalid for a disabled evaluation strategy.
        training_args_kwargs = dict(
            output_dir=str(output_dir),
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=max(1, self.config.logging_steps),
            eval_strategy=strategy,
            save_strategy="steps",
            save_steps=max(1, self.config.save_steps),
            gradient_accumulation_steps=max(1, self.config.gradient_accumulation_steps),
            report_to="none",
            dataloader_num_workers=0,
            save_total_limit=2,
        )
        if strategy == "steps":
            training_args_kwargs["eval_steps"] = max(1, self.config.eval_steps)

        trainer_args = TrainingArguments(**training_args_kwargs)

        trainer = Trainer(
            model=model,
            args=trainer_args,
            processing_class=tokenizer,
            data_collator=data_collator,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],
        )

        logger.info("Starting model training...")
        train_result = trainer.train()

        # Save the final model and tokenizer separately from intermediate checkpoints.
        model_output_dir = output_dir / "pegasus-samsum-model"
        tokenizer_output_dir = output_dir / "tokenizer"
        model_output_dir.mkdir(parents=True, exist_ok=True)
        tokenizer_output_dir.mkdir(parents=True, exist_ok=True)

        trainer.save_model(str(model_output_dir))
        tokenizer.save_pretrained(str(tokenizer_output_dir))

        logger.info(f"Trained model saved to: {model_output_dir}")
        logger.info(f"Tokenizer saved to: {tokenizer_output_dir}")
        return train_result


In [ ]:
# Diagnostic check: paths should be absolute and exist before training.
print("Project root:", PROJECT_ROOT)
print("Config path:", CONFIG_FILE_PATH, "->", CONFIG_FILE_PATH.is_file())
print("Params path:", PARAMS_FILE_PATH, "->", PARAMS_FILE_PATH.is_file())


Project root: D:\Somesh\NLP_Project
Config path: D:\Somesh\NLP_Project\config\config.yaml -> True
Params path: D:\Somesh\NLP_Project\params.yaml -> True


: 

In [ ]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
train_result = model_trainer.train()


[2026-10-24 10:29:36,465: INFO: common: yaml file: D:\Somesh\NLP_Project\config\config.yaml loaded successfully]
[2026-10-24 10:29:36,472: INFO: common: yaml file: D:\Somesh\NLP_Project\params.yaml loaded successfully]
[2026-10-24 10:29:36,474: INFO: common: created directory at: artifacts]
[2026-10-24 10:29:36,478: INFO: common: created directory at: artifacts/model_trainer]
[2026-10-24 10:29:36,481: INFO: 3788379127: Loading tokenizer and model: google/pegasus-cnn_dailymail]
[2026-10-24 10:29:36,811: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK"]
[2026-10-24 10:29:37,049: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-10-24 10:29:37,288: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTT

[2026-10-24 10:29:40,527: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-10-24 10:29:41,011: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/refs%2Fpr%2F12/model.safetensors "HTTP/1.1 302 Found"]


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 7978.53it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-10-24 10:29:49,704: INFO: 3788379127: Loading transformed dataset: artifacts\data_transformation\samsum_dataset]
[2026-10-24 10:29:49,826: INFO: 3788379127: Starting model training...]


d:\Somesh\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
